### Eine kleine Hilfe für die Verwendung von Gmsh ###

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gmsh
import helper_funcs.gmshtools as gm
%matplotlib widget

\
In gmshtools ist eine Netz-Klasse und nützliche Auswertungfunktionen definiert. Man kann Netze von denen man die Punkte und die Dreiecke und Kurvensegmente kennt oder gmsh-Modelle in die Klasse speichern. 

Zunächst ein einfaches handgemachtes Netz und wie man auf die Bestandteile der Instanz zugreift. Die Plotfunktion soll einfache schnelle Darstellungen ermöglichen. Jede Instanz erhält ein eigenes ax (Plotkoordinatensystem) zugewiesen. Die erhält man beim plotten auch als Rückgabeparameter zurück. Ohne weitere Angaben wird immer in diesen plot geplottet. Man kann allerdings auch in schon bestehende alte figures plotten wenn man ax kennt. Will man einen neuen plot machen, kann man eine neue figure erzeugen und dieses ax übergeben oder eine neue Instanz erzeugen.

Für das Plotten kann eine 3D oder 2D Achse gewählt werden. Default ist 3D, das lässt sich aber durch eine eigenschaft der Klasse (instanz.dim) von 3 auf 2 setzen.


In [ ]:
# einfaches eigenes Netz und eine "Kurve" dazu

p = np.array([[ 1, 0.7 ] , [ 0.5, 0.35 ] , [ 0.5, 0.18 ] , [ 0, 0 ] , [ 0, 0.7 ] , [ 1, 0 ]])
t = np.array([[ 0, 4, 1 ] , [ 3, 1, 4 ] , [ 5, 0, 1 ] , [ 3, 5, 2 ] , [ 2, 1, 3 ] , [ 5, 1, 2 ]], dtype=np.int64)
rand_seg = np.array([[ 3, 5 ] , [ 0, 4 ] , [ 4, 3 ],  [ 5, 0 ] ], dtype=np.int64)
innen_seg = np.array([[1,2]])

# punkte sind unter                            netz.nodes
# Die Elemente unter                           netz.Triangle.elements
# rand_seg unter dem key des dicts zu finden   netz.Rand.elements

netz1 = gm.MshHs(None, points=p, triangles=t, segments={'Rand': rand_seg, 'Innen': innen_seg})
# 3D-Plot
netz1.Triangle.plot(color='grey', alpha=0.3, node_label=True)
netz1.Innen.plot(color='red', alpha=1)
axx=netz1.Rand.plot(color='red', alpha=0.8)
axx.text(-0.5,1,0,"Dreiecke und Randkurve")
plt.show()

#2D plot
netz1 = gm.MshHs(None, points=p, triangles=t, segments={'Rand': rand_seg, 'Innen': innen_seg})
netz1.dim=2
netz1.Triangle.plot(color='blue', node_label=True)
netz1.Innen.plot(color='red',alpha=1)
axx=netz1.Rand.plot(color='red',alpha=1)
axx.text(0.5,1,"Dreiecke und Randkurve")
plt.show()



print("Segmente des Randes ",netz1.Rand.elements )
print("Knoten des Randes ",netz1.Rand.nodes )
print("Dreiecke, die an den Segmenten liegen ",netz1.Rand.connect)

In [ ]:
help(gm.MshHs)

\
Als nächstes Beispiel wird ein gmsh Modell erzeugt und die Klasse verwendet. Das Model ist sehr elementar aus Punkten und Linien aufgebaut, 
was eine sehr große Kontrolle über alle Bestandteile des Netzes erlaubt. Die Randkurven oder innen liegenden Kurven werden über physikalische 
Gruppen realisiert.

Beachte, die API von gmsh wird unter Umständen starten, die muss dann wieder geschlossen werden.

In [ ]:
gmsh.initialize()
name = 'TestGroups' 
gmsh.model.add(name)

#Äusseres Rechteck
P1=(-1,-1,0); P2=(4,-1,0); P3=(4,4,0); P4=(-1,4,0)
i1=gmsh.model.occ.addPoint(*P1)
i2=gmsh.model.occ.addPoint(*P2)
i3=gmsh.model.occ.addPoint(*P3)
i4=gmsh.model.occ.addPoint(*P4)
    
L1=gmsh.model.occ.addLine(i1,i2)
L2=gmsh.model.occ.addLine(i2,i3)
L3=gmsh.model.occ.addLine(i3,i4)
L4=gmsh.model.occ.addLine(i4,i1)

loop1 = gmsh.model.occ.addCurveLoop([L1,L2,L3,L4])

# Dreiecksloch
P5=(2.5,2,0); P6=(3.5,2,0); P7=(3.5,3,0);
i5=gmsh.model.occ.addPoint(*P5)
i6=gmsh.model.occ.addPoint(*P6)
i7=gmsh.model.occ.addPoint(*P7)
  
L5=gmsh.model.occ.addLine(i5,i6)
L6=gmsh.model.occ.addLine(i6,i7)
L7=gmsh.model.occ.addLine(i7,i5)

loop2 = gmsh.model.occ.addCurveLoop([L5,L6,L7])

# loop1 ist hauptrand, loop2 wird "abgezogen"
surf1= gmsh.model.occ.addPlaneSurface([loop1, loop2])



#Inneres Viereck
P10=(0,0,0); P11=(2,0,0); P12=(2,1,0); P13=(0,1,0)
i10=gmsh.model.occ.addPoint(*P10,tag=10)
i11=gmsh.model.occ.addPoint(*P11)
i12=gmsh.model.occ.addPoint(*P12)
i13=gmsh.model.occ.addPoint(*P13)

L10=gmsh.model.occ.addLine(i10,i11,tag=10)
L11=gmsh.model.occ.addLine(i11,i12)
L12=gmsh.model.occ.addLine(i12,i13)
L13=gmsh.model.occ.addLine(i13,i10)

gmsh.model.occ.synchronize()

# innere Linie soll später ins Netz integriert werden, deshalb embed
gmsh.model.mesh.embed(1,[L10,L11,L12,L13],2,surf1)

gmsh.model.occ.synchronize()

# Erzeuge Netz
mesh = gmsh.model.mesh.generate(2)


# Bilde nun Gruppen, z.B. Linien, die (bzw. zu zugehörigen knoten) 
# später verwendet werden sollen, Fläche immer dazutun

l0 = gmsh.model.addPhysicalGroup(1, [L5, L6, L7])
gmsh.model.setPhysicalName(1, l0, "RandDreieck")
    
l1 = gmsh.model.addPhysicalGroup(1, [L2,L4])
gmsh.model.setPhysicalName(1, l1, "Rand_D")

l2 = gmsh.model.addPhysicalGroup(1, [L10,L11])
gmsh.model.setPhysicalName(1, l2, "Innen_1")

f1 = gmsh.model.addPhysicalGroup(2, [surf1])
gmsh.model.setPhysicalName(2, f1, "Flaeche")


gmsh.option.setNumber("Mesh.SaveAll", 1)
gmsh.write(name+".brep")
gmsh.write(name+".msh")

try:
    gmsh.fltk.run()
except:
    print("Fehler bei gmsh.fltk.run()")  

# Lege Daten in der Klasse ab
netz2 = gm.MshHs(gmsh.model)

print("\nTeste Attribute von netz2, mit netz.")

gmsh.finalize()

netz2.Triangle.plot()
plt.show()
    
   

im nachfolgenden Fenster drücke man tab, dann werden für netz2 die Eigenschaften angezeigt. Wählt man z.B. Rand_D aus, kann man wieder mit tab die eiegenschaften/Methoden sehen.

In [ ]:
netz2.
netz2.Rand_D.


Das Netz kann gespeichert werden und dann auch wieder geladen werden.

In [ ]:
 #Zuerst vorangegangenen File laden
gmsh.initialize()
name = 'TestGroups' 
gmsh.open(name + ".msh")
netz2=gm.MshHs(gmsh.model)

netz2.Triangle.plot(color='grey', alpha=0.2)
netz2.Innen_1.plot(color='red', node_label=True, direction=True,alpha=0.9)
plt.show()

Und noch ein Netz, in dem Dreiecke und Vierecke vorliegen.

In [ ]:
gmsh.initialize()
    
name = 'bla' 
gmsh.model.add(name)
gmsh.option.setNumber("Mesh.CharacteristicLengthMin", 0.4)
gmsh.option.setNumber("Mesh.CharacteristicLengthMax", 0.4)

# Rechteck mit Rundungen kann auch damit gemacht werden (noch ein Parameter)
#gmsh.model.occ.addDisk(1.5,0.5,0,0.3,0.3,1)
gmsh.model.occ.addRectangle(0,0,0,1,1,10)
gmsh.model.occ.addRectangle(1,0,0,1,1,11)
gmsh.model.occ.fragment([(2, 10)], [(2, 11)])

gmsh.model.occ.synchronize()
#benutze rechtecke
gmsh.model.mesh.setRecombine(2, 11)
#vernetze
mesh = gmsh.model.mesh.generate(2)
gmsh.write(name+".msh")

try:
    gmsh.fltk.run()
except:
    print("Fehler bei gmsh.fltk.run()")  

netz3=gm.MshHs(gmsh.model)

netz3.Triangle.plot(color='blue', node_label=True)
netz3.Quadrilateral.plot(color='red', node_label=True)
plt.show()

gmsh.finalize()

\
Nun ein gmsh Modell bei dem elementare Geometrien durch erweiterte Objekte wie Rechtecke, Kreiseflächen, usw. gemacht werden. Auch das Ausschneiden wird 
mit einer anderen Vorgehensweise gemacht. Die Schnittkurve wird ermittelt. Zusätzlich wird das Netz auch verfeinert.

Auch hier wird geplottet, der Richtungssinn einer Kurve verändert, um mathematische positive Richtungen zu haben und dann wieder geplottet. Um zu verhindern dass es in den alten plot geht, wird eine neue Achse geöffnet und beim plot-Befehl ax=Achse übergeben. Man könnte auch in der Instanz die Achse überschreiben (netz4.ax=ax) oder einfach nocheinmal eine Instanz erzeugen. Der Richtungssinn kann mit der Methode flip geändert werden.

In [ ]:
gmsh.initialize()
    
name = 'bla' 
gmsh.model.add(name)

p0 = [0, 0, 0]; pr = [1.4, 0.5, 0]
#erstellt tags zu den punkten, den linien und ein flächentag
rechteck1 = gmsh.model.occ.addRectangle(*p0,2,1)
ellipse1 = gmsh.model.occ.addDisk(*pr, 0.4, 0.3)
gmsh.model.occ.synchronize()

# Schnittlinie
out = gmsh.model.getBoundary([(2, ellipse1)], oriented=False)
rand_innen = [e[1] for e in out] 
  
# Löcher bzw ausschneiden
out, out2 = gmsh.model.occ.cut([(2, rechteck1)],[(2, ellipse1)])
# (neues) Flächentag ermitteln 
flaeche_id = out[0][1]
    
# Innere Linie plazieren
p1 = [0.5, 0.2, 0]; p2 =[0.5, 0.8, 0]
pt1 = gmsh.model.occ.addPoint(*p1)
pt2 = gmsh.model.occ.addPoint(*p2)
l1 = gmsh.model.occ.addLine(pt1, pt2)

gmsh.model.occ.synchronize()
     
# Linie einbetten, damit die Netzknoten drauf liegen
gmsh.model.mesh.embed(1,[l1],2,flaeche_id)

# Jetzt gewünschte Kurven abspeichern
# Wichtig scheint zu sein auch immer die flaeche mit in die physikalische Gruppe zu stecken
sf = gmsh.model.addPhysicalGroup(2, [flaeche_id])
gmsh.model.setPhysicalName(2, sf, "Flaeche")

ri1 = gmsh.model.addPhysicalGroup(1, rand_innen )
gmsh.model.setPhysicalName(1, ri1, "RandInnen")    

ri2 = gmsh.model.addPhysicalGroup(1, [l1])
gmsh.model.setPhysicalName(1, ri2, "LinieInnen")


# noch eine Verfeinerung des netzes
dist_tag = gmsh.model.mesh.field.add("Distance")
gmsh.model.mesh.field.setNumbers(dist_tag, "EdgesList", [l1])       # zu diesen Linien wird der (minimale) Abstand berechnet
gmsh.model.mesh.field.setNumber(dist_tag, "Sampling", 100)          # auf den Linien werden dazu 100 pkt verwendet    

# abstandsfunktion verwenden
math_tag = gmsh.model.mesh.field.add("MathEval")
# die Formel h=0.02 + 0.3*Fi gibt die ungefähre mittlere Dreieckslänge an
gmsh.model.mesh.field.setString(math_tag, "F", "0.015 + 0.2*F%i"%(dist_tag))         # Fi = Distanzwert mit tag i
gmsh.model.mesh.field.setAsBackgroundMesh(math_tag)     

mesh = gmsh.model.mesh.generate(2)


# Und in die Klasse abspeichern.
netz4=gm.MshHs(gmsh.model)
# Man kann auch noch nachdem das Netz abgelegt wurde SubObjekte erzeugen
# z.B. aus den Line-Elementen nur bestimmte durch eine Funktion auswählen 
netz4.Line.select("DirichletRand", lambda x: (x[:, 0] < 1e-10) | (x[:, 1]<1e-10) )

# plotten
netz4.Triangle.plot(color='grey', alpha=0.2)
netz4.LinieInnen.plot(color='red')
netz4.RandInnen.plot(color='red', direction=True)
netz4.DirichletRand.plot(color='darkred', direction=True)
plt.show()


# Randrichtungen umkehren und wieder plotten. Da ein neuer plot entstehen soll wird eine neue Achse erzeugt.
# Dann muss diese Achse auch übergeben werden. 
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
# Man könnte auch netz4.ax=ax machen und dann die Aufrufe unten ohne ax=ax
netz4.Triangle.plot(ax=ax,color='grey', alpha=0.2)
netz4.RandInnen.flip(flip='right')
netz4.RandInnen.plot(ax=ax,color='darkred',direction=True)
plt.show()

gmsh.write(name+".msh")
gmsh.finalize()

Und ein 3D-Beispiel. Es ist zu beachten, dass in der Instanz auch Linien oder Dreiecke angelegt werden, die nicht über physikalische Gruppen 
erzeugt wurden. Sie entstehen durch die geometrische Objektverwaltung von gmsh. Das erste Bild zeigt welche Objekte automatisch erzeugt wurden in 
diesem Beispiel. Es sind letztlich die Ränder der definierten geometrischen Objekte (Würfel, Fläche 1 und Fläche 2, Linien, usw.). Dies war auch schon bei den anderen gmsh-Modellen weiter oben der Fall, sie wurden nur nicht gezeichnet, sie befinden sich aber in den Instanzen.

Das zweite Bild zeigt dann die Tetraeder-Elemente.

In [ ]:
gmsh.initialize()
gmsh.model.add("3dBsp")

# Großer Würfel (Startpunkt 0,0,0 – Größe 2×2×2)
big = gmsh.model.occ.addBox(0, 0, 0, 2, 2, 2)

# innere Punkte
p1 = gmsh.model.occ.addPoint(0.5, 0.5, 0.5)
p2 = gmsh.model.occ.addPoint(1.5, 0.5, 0.5)
p3 = gmsh.model.occ.addPoint(1.5, 1.5, 0.5)
p4 = gmsh.model.occ.addPoint(0.5, 1.5, 0.5)

p5 = gmsh.model.occ.addPoint(1.5, 0.5, 1)
p6 = gmsh.model.occ.addPoint(1.5, 1.5, 1)

# innere Linien
l1 = gmsh.model.occ.addLine(p1, p2)
l2 = gmsh.model.occ.addLine(p2, p3)
l3 = gmsh.model.occ.addLine(p3, p4)
l4 = gmsh.model.occ.addLine(p4, p1)
l5 = gmsh.model.occ.addLine(p2, p5)
l6 = gmsh.model.occ.addLine(p5, p6)
l7 = gmsh.model.occ.addLine(p6, p3)


# innere Flächen
loop1 = gmsh.model.occ.addCurveLoop([l1, l2, l3, l4])
loop2 = gmsh.model.occ.addCurveLoop([l5, l6, l7, -l2])

# Fläche erzeugen
surf1 = gmsh.model.occ.addPlaneSurface([loop1])
surf2 = gmsh.model.occ.addPlaneSurface([loop2])

gmsh.model.occ.synchronize()

# Flächen einbetten
gmsh.model.mesh.embed(2,[surf1, surf2],3,big)

# in 2D wars
#gmsh.model.mesh.embed(1,[L10,L11,L12,L13],2,surf1)

gmsh.model.mesh.generate(3)
gmsh.write("3dBsp.msh")

try:
    gmsh.fltk.run()
except:
    print("Fehler bei gmsh.fltk.run()")  



netz5=gm.MshHs(gmsh.model)

netz5.Triangle.plot(color='blue', alpha=0.7)
netz5.Line.plot(color='red', alpha=0.4, node_label=True)
plt.show()

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
netz5.Triangle.plot(ax=ax,color='blue', alpha=0.3)
netz5.Tetrahedron.plot(ax=ax,color='grey', alpha=0.2)
plt.show()

gmsh.finalize()

Noch eine weitere Geometrie, mit der die Normalenableitung und der Gradient erklärt wird. Es geht also hier um Auswertungen, falls die Lösung 
bekannt ist. Versuchen Sie das Netz selbst einmal mit gmsh zu erzeugen.

In [ ]:
#np.savetxt('GmshExample_points.dat', poi)
#np.savetxt('GmshExample_phis.dat', phi)
#np.savetxt('GmshExample_elements.dat', tri, fmt='%i')
#np.savetxt('GmshExample_bound.dat', np.array(bseg[0]), fmt='%i')
#np.savetxt('GmshExample_inner.dat', np.array(cseg[0]), fmt='%i')

# Ein Netz welches über einfache Listen definiert ist 
# (z.B. mit anderem programm definiert).
# Wird verwendet um verschiedene Auswertedinge zu testen

poi = np.loadtxt('GmshExample_points.dat')
phi = np.loadtxt('GmshExample_phis.dat')
tri = np.loadtxt('GmshExample_elements.dat', dtype=np.int64)
# Randkurve
bseg = np.loadtxt('GmshExample_bound.dat', dtype=np.int64)
# Innere Kurve
iseg = np.loadtxt('GmshExample_inner.dat', dtype=np.int64)    

# Schreibe Netz in MshHs Klasse:
netz6 = gm.MshHs(None, points=poi, triangles=tri, segments={'KurveRand': bseg, 'KurveInnen': iseg})
#netz.Triangle.plot(color='blue', alpha=0.3, node_label=True)
#plt.show()

plt.figure()
#plots, alternativ auch mit net.Triangle.plot()
plt.triplot(poi[:,0],poi[:,1],tri,color='gray')
plt.title("Erzeuge dieses Netz mit gmsh, Parabel: y=7-(x-5)^2")
plt.show()

# Ergebnis plotten
poi = netz6.points
tri = netz6.Triangle.elements
plt.figure()
plt.tricontourf(poi[:,0],poi[:,1],tri,phi,levels=30,cmap='jet')
plt.tricontour(poi[:,0],poi[:,1],tri,phi,levels=30,colors='black')
plt.xlim((-0.2,10.2))
plt.ylim((-0.2,8.2));
plt.show()    



    
# Normalenableitung mit MshHs, ergebnis
netz6.Triangle.plot()
netz6.KurveInnen.plot(color='red',alpha=1)
netz6.KurveRand.plot(color='red',alpha=0.8)
plt.show()

res_rand = gm.NormalDerivative('KurveRand', netz6, phi, normal_direction='left')
res_innen_r = gm.NormalDerivative('KurveInnen', netz6, phi)
res_innen_l = gm.NormalDerivative('KurveInnen', netz6, phi, normal_direction='left')




#plot normal derivative
plt.figure()
plt.plot(res_rand[0],res_rand[1],'o',markersize=8,label='boundary')
plt.plot(res_innen_r[0],res_innen_r[1],'o',markersize=8,label='inner, right')
plt.plot(res_innen_l[0],res_innen_l[1],'o',markersize=8,label='inner_left')

#indicate different parts of boundary
s0 = res_rand[0]
plt.plot([2,2],[0,10],'k--')
plt.plot([s0[-1]-10,s0[-1]-10],[-2,5],'k--')
plt.plot([s0[-1]-12,s0[-1]-12],[-5,5],'k--')
plt.grid()

plt.xlabel("curve length s")
plt.ylabel(r"normal derivative $\;\;\frac{\partial \Phi}{\partial n}$")


# theory, boundary
R = 2
#first part
t=np.linspace(8,6,100)
s=8-t
dF=t
plt.plot(s,dF,lw=2,color="cyan",label="Theory")

# second part
t=np.linspace(np.pi/2,-np.pi/2,100)
s=2+(np.pi/2-t)*R
dF=4*np.sin(t)*np.cos(t) + 4*np.cos(t) + 3*np.sin(t)
plt.plot( s,dF,lw=2,color="cyan" )

#third part (circle)
t=np.linspace(2,0,100)
s=2+np.pi*R+2-t
dF=t
plt.plot(s,dF,lw=2,color="cyan")

#fourth part
t=np.linspace(0,10,100)
s=4+np.pi*R+t
dF=(t+3)
plt.plot(s,dF,lw=2,color="cyan")

#theory inner curve
t=np.linspace(2.5,7,500)
# x(t)
xx=t; yy=-(t-5)**2+7
# ds
ds=np.sqrt(np.diff(xx)**2+np.diff(yy)**2)
# running length of curve
s=np.append(np.array([0]),np.cumsum(ds))

#left and right depends at inner curves on the orientation of the curve (starting point)
# take phi on left side of curve, normal points to the right
# grad(phi)*n
dF=(yy*2*(xx-5) + (xx+3)*1)/(np.sqrt(4*(xx-5)**2+1))
plt.plot(s,dF,lw=2,color="cyan")

# take phi on the right side of curve, normal points to the left
# grad(phi)*n
dF= -( (yy+6*(xx-5))*2*(xx-5) + (xx+3+3)*1)/(np.sqrt(4*(xx-5)**2+1))
plt.plot(s,dF,lw=2,color="cyan")


plt.legend()
plt.show()

#Gradient plotten
plt.figure()
pp = netz6.points[:,[0, 1]]
xx,yy,ggx,ggy=gm.ComputeGradient(pp,netz6.Triangle.elements,phi,location='middle' )
plt.triplot(pp[:, 0], pp[:, 1], netz6.Triangle.elements)
plt.quiver(xx,yy, ggx,ggy, color='green',scale=500)
plt.show()

print("Ende")

##### Problem:

Compute numerically an approximation of

$$
(i) \;\;\int\limits_0^{14+2\pi} (x+1)(y+2)\frac{\partial \Phi}{\partial n}\Bigg|_{boundary} ds \qquad\qquad 
(ii)\;\;\int\limits_{Inner \; Curve} x\left(\frac{\partial \Phi}{\partial n}\Bigg|_{right} + \frac{\partial \Phi}{\partial n}\Bigg|_{left}\right) ds
$$

using scipy.integrate.simps()

Result: 

(i) -1550.539

(ii) 430.2717758

##### Problem:

Compute numerically an approximation of

$$
(i) \;\;\int\limits_{inner\, right} (x+y)\left(\; -\frac{\partial \Phi}{\partial y} \;,\; 4\frac{\partial \Phi}{\partial x} \;\right)\cdot\vec{n}\; ds \qquad\qquad 
(ii) \;\;\int\limits_{inner\, left} \begin{pmatrix} a & b\\b &-a \end{pmatrix}\cdot\vec{n}\;ds \quad \mbox{with}\;\; a=\left(\frac{\partial \Phi}{\partial x}\right)^2-\left(\frac{\partial \Phi}{\partial y}\right)^2 \;\;
\mbox{and} \;\; b=\frac{\partial \Phi}{\partial x}\frac{\partial \Phi}{\partial y}
$$

using scipy.integrate.simps()

Result: 

(i) 876.2984

(ii) [-106.499835 , -192.0202299]

Noch ein kleine Hilfe, um zu erkennen, ob Gebiete richtig zugewiesen wurden.

$$
f=\begin{cases}
1, & \text{wenn } x < 2 \\
3, & \text{wenn } x >8 \wedge y<4 \\
0, & \text{sonst } 
\end{cases}
$$


In [ ]:
# define function on triangles

# all triangle center
pp=np.sum(poi[tri],axis=1)/3.
vals=np.zeros(len(tri))
vals[pp[:,0]<2] = 1
vals[(pp[:,0]>8) & (pp[:,1]<4)] = 3

plt.figure()
plt.tripcolor(poi[:,0],poi[:,1],tri,vals,cmap='jet')
plt.colorbar(orientation='horizontal')
plt.show()